# Python程序内存优化方法总结

## 1. **数据结构优化**

### 紧凑数据类型

In [ ]:
# 使用array代替list
import array
# 整数数组
arr = array.array('i', [1, 2, 3, 4])  # 'i'表示4字节整数
arr = array.array('h', [1, 2, 3, 4])  # 'h'表示2字节整数
arr = array.array('b', [1, 2, 3, 4])  # 'b'表示1字节整数

# 浮点数数组
arr = array.array('f', [1.0, 2.0, 3.0])  # 'f'表示4字节浮点数


### 高效布尔数组

In [ ]:
# 使用bytearray代替bool列表
visited = bytearray(1000)  # 每个元素1字节
visited = bitarray(1000)   # 每个元素1位（需要安装bitarray）

# 使用位运算手动管理
visited = 0  # 用整数位表示状态
def set_visited(pos):
    global visited
    visited |= (1 << pos)


## 2. **内存高效的数据结构**

### 使用生成器

In [ ]:
# 避免创建完整列表
def read_large_file(filename):
    with open(filename) as f:
        for line in f:
            yield int(line.strip())

# 使用
for number in read_large_file('large.txt'):
    process(number)


### 使用字典视图

In [ ]:
data = {'a': 1, 'b': 2, 'c': 3}
# 不创建新列表
for key in data.keys():    # 创建键列表（浪费内存）
for key in data:           # 使用迭代器（推荐）


## 3. **算法优化**

### 避免重复计算和存储

In [ ]:
# 坏：重复计算
for i in range(len(data)):
    for j in range(len(data)):
        result = expensive_calc(data[i], data[j])

# 好：预计算或使用缓存
precomputed = [expensive_calc(x) for x in data]


### 及时释放内存

In [ ]:
# 及时删除大对象
large_data = load_huge_dataset()
process(large_data)
del large_data  # 立即释放
import gc
gc.collect()    # 强制垃圾回收


## 4. **数值计算优化**

### 使用NumPy（科学计算）

In [ ]:
import numpy as np

# NumPy数组比Python列表更紧凑
arr = np.array([1, 2, 3], dtype=np.int32)    # 4字节/元素
arr = np.array([1, 2, 3], dtype=np.int16)    # 2字节/元素
arr = np.array([1, 2, 3], dtype=np.int8)     # 1字节/元素


### 使用Pandas高效数据类型

In [ ]:
import pandas as pd

# 优化数据类型
df['col'] = df['col'].astype('int32')    # 代替默认int64
df['col'] = df['col'].astype('category') # 分类数据


## 5. **文件处理优化**

### 逐行处理大文件

In [ ]:
# 坏：一次性加载
with open('large.txt') as f:
    data = f.readlines()  # 可能内存爆炸

# 好：逐行处理
with open('large.txt') as f:
    for line in f:
        process(line)


### 使用内存映射

In [ ]:
import mmap

with open('large_file.bin', 'r+b') as f:
    with mmap.mmap(f.fileno(), 0) as mm:
        # 像操作内存一样操作文件
        data = mm[1000:2000]


## 6. **对象优化**

### 使用`__slots__`

In [ ]:
class LargeClass:
    __slots__ = ['x', 'y', 'z']  # 固定属性，节省内存
    def __init__(self, x, y, z):
        self.x = x
        self.y = y
        self.z = z


### 使用namedtuple或dataclass

In [ ]:
from collections import namedtuple
from dataclasses import dataclass

# namedtuple（不可变）
Point = namedtuple('Point', ['x', 'y'])

# dataclass（Python 3.7+）
@dataclass
class Point:
    x: int
    y: int


## 7. **编码和压缩技术**

### 位置编码

In [ ]:
# 将二维坐标编码为一维
def encode(i, j, width):
    return i * width + j

def decode(pos, width):
    return pos // width, pos % width


### 数据压缩

In [ ]:
import zlib

# 压缩重复数据
data = "重复的文本" * 1000
compressed = zlib.compress(data.encode())


## 8. **内存监控和分析**

### 监控内存使用

In [ ]:
import psutil
import os

def memory_usage():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 / 1024  # MB

print(f"内存使用: {memory_usage()} MB")


### 使用memory_profiler

In [ ]:
# 安装: pip install memory_profiler
from memory_profiler import profile

@profile
def memory_intensive_function():
    # 函数代码
    pass


## 9. **特定场景优化技巧**

### 图搜索问题

In [ ]:
# 使用位置编码和最小代价数组
def dijkstra_optimized(n, m, grid):
    visited = bytearray(n * m)
    min_cost = [float('inf')] * (n * m)
    
    def encode(i, j):
        return i * m + j
    
    # ... 其余算法逻辑


### 大数据处理

In [ ]:
# 分批处理
def process_in_chunks(data, chunk_size=1000):
    for i in range(0, len(data), chunk_size):
        chunk = data[i:i + chunk_size]
        process_chunk(chunk)
        del chunk  # 及时释放


## 10. **系统级优化**

### 使用64位Python处理大内存
- 32位Python有内存限制（约2-4GB）
- 64位Python可以访问更多内存

### 调整垃圾回收

In [ ]:
import gc

# 调整垃圾回收阈值
gc.set_threshold(700, 10, 10)

# 禁用自动垃圾回收（谨慎使用）
gc.disable()
# ... 关键代码段
gc.enable()
gc.collect()


这些方法可以根据具体应用场景组合使用，在内存使用和性能之间找到最佳平衡点。